<a href="https://colab.research.google.com/github/parthdabhi5195/YT-AI/blob/main/YT_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os

DATA_PATH = "/Users/parth/Desktop/Youtube/YT AI/Synthetic Training Data/story-pipeline v3/data/clean/final/stories.jsonl"
OUT_DIR = "slm_out"
LIMIT = 50000


In [12]:
!pip3 install tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 7.2 MB/s  0:00:00 eta 0:00:01


## STEP 1: Importing Dataset


In [8]:
import json, random

stories = [json.loads(line)["story_text"] for line in open(DATA_PATH)]


In [51]:
print(stories[:2])

['Have you ever been accused of something you absolutely didn\'t do? My RV trip with my parents, Joel and Aurora, was supposed to be a peaceful escape, but it turned into a nightmare when officers boarded. They’d tracked me down for forging Javier’s signature on a property deed – a deed he swore he never authorized.\n\nThey hauled me into a cramped interrogation room, the air thick with suspicion. The evidence was damning: a scanned signature, eerily similar to mine, and Javier, my second cousin I was meeting for the first time, insisted it was me. My parents, Laura and Joel, arrived, their faces etched with worry as they watched me squirm under the officer\'s relentless gaze.\n\nThen, the truth hit me. Javier’s been bragging about his uncanny ability to mimic signatures, a skill he’d honed practicing for his code enforcement job. I looked at him, his eyes wide, and knew. "It wasn\'t me," I declared, my voice shaking but firm. "It was Javier. He’s the one with the forged signature beca

In [9]:
if LIMIT:
    stories = stories[:LIMIT] # get only a small subset

random.seed(42)
random.shuffle(stories) # randomly shuffle subset.

n_val = int(len(stories) * 0.005)
n_test = int(len(stories) * 0.005)

val = stories[: n_val]
test = stories[n_val : n_test + n_val]
train = stories[n_test + n_val :]

print(f"val len : {len(val)}")
print(f"test len : {len(test)}")
print(f"train len : {len(train)}")

val len : 250
test len : 250
train len : 49500


## STEP 2: Tokenize the Dataset

(1) Tokenize the story dataset into tokenIDs

(2) Append <|endoftext|> after each story

(3) Create a file called "train.bin", "val.bin", "test.bin" to store all tokenIDs from the entire dataset. This is to increase speed (byte files don't need to parse characters like they have to in JSON), decrease storage size, and allow memory-mapping (letting us work with files larger than our RAM size by retrieving only active chunks).
(4) Make sure tokenIDs are stored to disk rather than RAM.

In [ ]:
import tiktoken
import numpy as np
from tqdm.auto import tqdm

enc = tiktoken.get_encoding("gpt2")
eot_id = enc.eot_token # <|endoftext|>


<class 'tiktoken.core.Encoding'>


In [52]:
# what encoding looks like
tokens = enc.encode(stories[1])
individual_tokens = [enc.decode([uid]) for uid in tokens]

print(tokens)
print("Split Text Chunks:", individual_tokens)

[3666, 5101, 547, 25711, 13, 632, 2936, 588, 257, 12840, 11, 4692, 6147, 18894, 656, 616, 38163, 11, 290, 314, 3521, 447, 247, 83, 772, 766, 508, 373, 1804, 340, 13, 198, 198, 6423, 11, 257, 3809, 314, 2993, 11, 257, 3809, 314, 447, 247, 67, 13467, 329, 812, 11, 2005, 832, 262, 13619, 13, 29721, 6204, 625, 502, 11, 465, 1986, 555, 46155, 13, 366, 42516, 553, 339, 2540, 11, 465, 3809, 1877, 11, 366, 3003, 750, 345, 7808, 340, 1701, 314, 599, 46322, 11, 2111, 284, 1319, 24082, 1479, 11, 564, 250, 38518, 644, 30, 1867, 389, 345, 3375, 546, 30, 447, 251, 29721, 9514, 616, 21289, 11, 465, 17841, 1327, 3101, 13, 366, 464, 28637, 13, 383, 6617, 326, 674, 1641, 1363, 318, 41802, 6164, 11, 407, 12431, 13, 7157, 340, 625, 11, 393, 428, 3011, 257, 1256, 4785, 526, 198, 198, 5960, 30052, 11, 314, 5636, 5263, 11, 19642, 10666, 268, 871, 11, 2111, 284, 10014, 790, 7104, 299, 566, 287, 428, 3027, 27123, 2119, 13, 314, 12086, 262, 9155, 4314, 3526, 287, 262, 1468, 15857, 563, 11, 257, 1295, 339, 447, 